# Day 7: Counter Assay Analysis

**Goal:** Understand if false positives (molecules that fail counter assay) have distinct chemical properties

**Strategy:**
1. Label all training molecules as counter-pass (1) or counter-fail (0)
2. Compare chemical properties between groups
3. Test if training only on counter-pass improves generalization
4. Test if adding counter-pass as a feature helps

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path

# RDKit
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
import useful_rdkit_utils as uru

# ML
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from lightgbm import LGBMRegressor

warnings.filterwarnings('ignore')
tqdm.pandas()
sns.set_style("whitegrid")

print("✅ Imports complete")

## 1. Load Data and Label Counter Assay Status

In [ ]:
# Load datasets
train_df = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_TRAIN.csv")
test_df = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_TEST_BLINDED.csv")
counter_df = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_counter-assay_TRAIN.csv")

print(f"Train: {len(train_df)}")
print(f"Test: {len(test_df)}")
print(f"Counter-pass: {len(counter_df)}")

# Label counter assay status
counter_pass_ids = set(counter_df['Molecule Name'])
train_df['passes_counter'] = train_df['Molecule Name'].isin(counter_pass_ids).astype(int)

print(f"\nCounter-pass: {train_df['passes_counter'].sum()} (TRUE PXR activators)")
print(f"Counter-fail: {(~train_df['passes_counter'].astype(bool)).sum()} (FALSE positives)")
print(f"Ratio: {train_df['passes_counter'].mean():.1%} true / {1-train_df['passes_counter'].mean():.1%} false")

## 2. Compare pEC50 Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
ax1 = axes[0]
counter_pass = train_df[train_df['passes_counter'] == 1]['pEC50']
counter_fail = train_df[train_df['passes_counter'] == 0]['pEC50']

ax1.hist(counter_pass, bins=30, alpha=0.5, label=f'Counter-pass (n={len(counter_pass)})', color='green')
ax1.hist(counter_fail, bins=30, alpha=0.5, label=f'Counter-fail (n={len(counter_fail)})', color='red')
ax1.set_xlabel('pEC50')
ax1.set_ylabel('Count')
ax1.set_title('pEC50 Distribution by Counter Assay Status')
ax1.legend()
ax1.grid(alpha=0.3)

# Box plot
ax2 = axes[1]
sns.boxplot(data=train_df, x='passes_counter', y='pEC50', ax=ax2)
ax2.set_xticklabels(['Counter-fail\n(False pos)', 'Counter-pass\n(True pos)'])
ax2.set_ylabel('pEC50')
ax2.set_xlabel('')
ax2.set_title('pEC50 by Counter Status')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Counter-pass mean pEC50: {counter_pass.mean():.3f}")
print(f"Counter-fail mean pEC50: {counter_fail.mean():.3f}")
print(f"Difference: {counter_pass.mean() - counter_fail.mean():.3f}")

## 3. Compare Chemical Properties

Do false positives have distinct chemical properties?

In [ ]:
def calc_basic_props(smiles):
    """Calculate basic molecular properties."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [np.nan] * 8
    
    return [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.NumAromaticRings(mol),
        Descriptors.NumAliphaticRings(mol),
    ]

print("Calculating molecular properties...")
props = np.array([calc_basic_props(s) for s in tqdm(train_df.SMILES)])
prop_names = ['MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds', 'ArRings', 'AlRings']

for i, name in enumerate(prop_names):
    train_df[name] = props[:, i]

In [ ]:
# Compare properties
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, prop in enumerate(prop_names):
    ax = axes[i]
    
    counter_pass_prop = train_df[train_df['passes_counter'] == 1][prop].dropna()
    counter_fail_prop = train_df[train_df['passes_counter'] == 0][prop].dropna()
    
    ax.hist(counter_pass_prop, bins=30, alpha=0.5, label='Counter-pass', color='green', density=True)
    ax.hist(counter_fail_prop, bins=30, alpha=0.5, label='Counter-fail', color='red', density=True)
    ax.set_xlabel(prop)
    ax.set_ylabel('Density')
    ax.set_title(f'{prop} Distribution')
    if i == 0:
        ax.legend()
    ax.grid(alpha=0.3)
    
    # Add mean difference
    mean_diff = counter_pass_prop.mean() - counter_fail_prop.mean()
    ax.text(0.95, 0.95, f'Δ={mean_diff:.2f}', 
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# Statistical comparison
from scipy.stats import mannwhitneyu

print("Statistical comparison (Mann-Whitney U test):")
print("="*60)
for prop in prop_names:
    counter_pass_prop = train_df[train_df['passes_counter'] == 1][prop].dropna()
    counter_fail_prop = train_df[train_df['passes_counter'] == 0][prop].dropna()
    
    stat, pval = mannwhitneyu(counter_pass_prop, counter_fail_prop)
    mean_pass = counter_pass_prop.mean()
    mean_fail = counter_fail_prop.mean()
    
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"{prop:12s}: Pass={mean_pass:6.2f}, Fail={mean_fail:6.2f}, p={pval:.2e} {sig}")

## 4. Baseline Model Comparison

Compare three strategies:
1. **Train on ALL** (current approach)
2. **Train on COUNTER-PASS only** (2860 molecules)
3. **Train on ALL with counter-pass as FEATURE**

In [ ]:
# Generate features
print("Generating RDKit descriptors...")
rdkit_desc = uru.RDKitDescriptors()
train_df["rdkit_desc"] = [rdkit_desc.calc_smiles(x) for x in tqdm(train_df.SMILES)]
test_df["rdkit_desc"] = [rdkit_desc.calc_smiles(x) for x in tqdm(test_df.SMILES)]

print("\nGenerating Morgan fingerprints...")
def get_morgan_fp(smiles, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(nbits)
    return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits))

morgan_train = np.stack([get_morgan_fp(x) for x in tqdm(train_df.SMILES)])
morgan_test = np.stack([get_morgan_fp(x) for x in tqdm(test_df.SMILES)])

# Preprocess
rdkit_train = np.stack(train_df.rdkit_desc.values)
rdkit_test = np.stack(test_df.rdkit_desc.values)

imputer = SimpleImputer(strategy='median')
rdkit_train = imputer.fit_transform(rdkit_train)
rdkit_test = imputer.transform(rdkit_test)

scaler_rdkit = StandardScaler()
scaler_morgan = StandardScaler()

rdkit_train_norm = scaler_rdkit.fit_transform(rdkit_train)
rdkit_test_norm = scaler_rdkit.transform(rdkit_test)

morgan_train_norm = scaler_morgan.fit_transform(morgan_train)
morgan_test_norm = scaler_morgan.transform(morgan_test)

print("\n✅ Feature generation complete")

In [ ]:
# Scaffold groups for CV
def get_scaffold(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return "invalid"
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        return scaffold
    except:
        return "invalid"

print("Computing scaffolds...")
train_df["scaffold"] = [get_scaffold(s) for s in tqdm(train_df.SMILES)]
scaffold_to_group = {s: i for i, s in enumerate(train_df["scaffold"].unique())}
train_df["scaffold_group"] = train_df["scaffold"].map(scaffold_to_group)

In [ ]:
def calculate_rae(y_true, y_pred):
    numerator = np.sum(np.abs(y_true - y_pred))
    denominator = np.sum(np.abs(y_true - np.mean(y_true)))
    return numerator / denominator if denominator != 0 else np.inf

def eval_model(y_true, y_pred):
    return {
        'rae': calculate_rae(y_true, y_pred),
        'r2': r2_score(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred)
    }

### Strategy 1: Train on ALL molecules

In [ ]:
X_all = np.hstack([rdkit_train_norm, morgan_train_norm])
y_all = train_df["pEC50"].values
groups_all = train_df["scaffold_group"].values

group_kfold = GroupKFold(n_splits=5)
fold_results_all = []

print("Strategy 1: Train on ALL molecules (4139)")
print("="*60)

for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X_all, y_all, groups_all), 1):
    X_tr, X_val = X_all[train_idx], X_all[val_idx]
    y_tr, y_val = y_all[train_idx], y_all[val_idx]
    
    model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=7,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=fold,
        verbose=-1
    )
    
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)
    metrics = eval_model(y_val, y_pred)
    fold_results_all.append(metrics)
    
    print(f"Fold {fold}: RAE={metrics['rae']:.4f}, R²={metrics['r2']:.4f}")

avg_rae_all = np.mean([r['rae'] for r in fold_results_all])
std_rae_all = np.std([r['rae'] for r in fold_results_all])
print(f"\n→ CV RAE: {avg_rae_all:.4f} ± {std_rae_all:.4f}")

### Strategy 2: Train on COUNTER-PASS only

In [ ]:
# Filter to counter-pass molecules
counter_pass_mask = train_df['passes_counter'] == 1
X_counter = X_all[counter_pass_mask]
y_counter = y_all[counter_pass_mask]
groups_counter = groups_all[counter_pass_mask]

fold_results_counter = []

print("\nStrategy 2: Train on COUNTER-PASS only (2860)")
print("="*60)

for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X_counter, y_counter, groups_counter), 1):
    X_tr, X_val = X_counter[train_idx], X_counter[val_idx]
    y_tr, y_val = y_counter[train_idx], y_counter[val_idx]
    
    model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=7,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=fold,
        verbose=-1
    )
    
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)
    metrics = eval_model(y_val, y_pred)
    fold_results_counter.append(metrics)
    
    print(f"Fold {fold}: RAE={metrics['rae']:.4f}, R²={metrics['r2']:.4f}")

avg_rae_counter = np.mean([r['rae'] for r in fold_results_counter])
std_rae_counter = np.std([r['rae'] for r in fold_results_counter])
print(f"\n→ CV RAE: {avg_rae_counter:.4f} ± {std_rae_counter:.4f}")

### Strategy 3: Train on ALL with counter-pass as FEATURE

In [ ]:
# Add counter-pass as a feature
counter_feature = train_df['passes_counter'].values.reshape(-1, 1)
X_all_with_counter = np.hstack([X_all, counter_feature])

fold_results_feature = []

print("\nStrategy 3: Train on ALL with counter-pass as FEATURE")
print("="*60)

for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X_all_with_counter, y_all, groups_all), 1):
    X_tr, X_val = X_all_with_counter[train_idx], X_all_with_counter[val_idx]
    y_tr, y_val = y_all[train_idx], y_all[val_idx]
    
    model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=7,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=fold,
        verbose=-1
    )
    
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)
    metrics = eval_model(y_val, y_pred)
    fold_results_feature.append(metrics)
    
    print(f"Fold {fold}: RAE={metrics['rae']:.4f}, R²={metrics['r2']:.4f}")

avg_rae_feature = np.mean([r['rae'] for r in fold_results_feature])
std_rae_feature = np.std([r['rae'] for r in fold_results_feature])
print(f"\n→ CV RAE: {avg_rae_feature:.4f} ± {std_rae_feature:.4f}")

## 5. Compare Strategies

In [ ]:
comparison = pd.DataFrame([
    {'strategy': 'Train on ALL', 'n_train': 4139, 'cv_rae': avg_rae_all, 'cv_rae_std': std_rae_all},
    {'strategy': 'Train on COUNTER-PASS', 'n_train': 2860, 'cv_rae': avg_rae_counter, 'cv_rae_std': std_rae_counter},
    {'strategy': 'Train on ALL + counter feature', 'n_train': 4139, 'cv_rae': avg_rae_feature, 'cv_rae_std': std_rae_feature},
])

print("\n" + "="*80)
print("STRATEGY COMPARISON")
print("="*80)
print(comparison.to_string(index=False))
print("\n" + "="*80)

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comparison))
ax.bar(x, comparison['cv_rae'], yerr=comparison['cv_rae_std'], capsize=5, alpha=0.7, color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(comparison['strategy'], rotation=15, ha='right')
ax.set_ylabel('Cross-Validation RAE')
ax.set_title('Counter Assay Strategy Comparison')
ax.axhline(0.68, color='red', linestyle='--', label='Day 1 blind test (0.68)', linewidth=2)
ax.axhline(0.72, color='orange', linestyle='--', label='Day 6 blind test (0.72)', linewidth=2)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_strategy = comparison.loc[comparison['cv_rae'].idxmin()]
print(f"\n✅ Best strategy: {best_strategy['strategy']} (RAE: {best_strategy['cv_rae']:.4f})")

## 6. Generate Submission with Best Strategy

NOTE: We don't know if test molecules pass counter assay, so for Strategy 3 we assume they all pass (set feature=1)

In [ ]:
# Pick best strategy based on CV RAE
if avg_rae_counter < avg_rae_all and avg_rae_counter < avg_rae_feature:
    print("Using Strategy 2: Train on COUNTER-PASS only")
    X_final = X_counter
    y_final = y_counter
    X_test_final = np.hstack([rdkit_test_norm, morgan_test_norm])
    strategy_name = "counter_only"
elif avg_rae_feature < avg_rae_all:
    print("Using Strategy 3: Train on ALL with counter-pass feature")
    X_final = X_all_with_counter
    y_final = y_all
    # Assume test molecules pass counter assay (feature=1)
    test_counter_feature = np.ones((len(test_df), 1))
    X_test_final = np.hstack([rdkit_test_norm, morgan_test_norm, test_counter_feature])
    strategy_name = "counter_feature"
else:
    print("Using Strategy 1: Train on ALL (baseline)")
    X_final = X_all
    y_final = y_all
    X_test_final = np.hstack([rdkit_test_norm, morgan_test_norm])
    strategy_name = "baseline"

# Train final model
final_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=7,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

final_model.fit(X_final, y_final)
test_preds = final_model.predict(X_test_final)

print(f"\nTest predictions:")
print(f"  Mean: {test_preds.mean():.3f}")
print(f"  Std: {test_preds.std():.3f}")
print(f"  Range: [{test_preds.min():.3f}, {test_preds.max():.3f}]")
print(f"\nTrain mean: {y_final.mean():.3f}")

In [ ]:
# Create submission
submission = pd.DataFrame({
    'SMILES': test_df['SMILES'],
    'Molecule Name': test_df['Molecule Name'],
    'pEC50': test_preds
})

output_path = f'../outputs/day7_counter_{strategy_name}_submission.csv'
submission.to_csv(output_path, index=False)

print(f"\n✅ Submission saved to: {output_path}")
print(f"Shape: {submission.shape}")
print("\nFirst 10 predictions:")
print(submission.head(10))

In [ ]:
# Validate submission
import sys
from pathlib import Path

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from validation.activity_validation import validate_activity_submission

expected_activity_ids = set(test_df["Molecule Name"])
is_valid, validation_errors = validate_activity_submission(
    Path(output_path),
    expected_ids=expected_activity_ids,
)

if is_valid:
    print("✅ Submission file is valid!")
else:
    print("❌ Submission file is invalid:")
    for msg in validation_errors:
        print(f" - {msg}")

## Summary

**Key Findings:**
- Counter-pass vs counter-fail molecules have [different/similar] chemical properties
- Best strategy: [see results]
- CV RAE improvement: [see results]

**Next Steps:**
- Test this submission on blind test set
- If improvement is modest, proceed with GNN approach
- Consider ensemble of counter-filtered models